# Load Packages

In [9]:
# usuals
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split

# Load Datasets

In [74]:
AADT_df = pd.read_csv('Annual_Average_Daily_Traffic.csv')
display(AADT_df.head())
display(AADT_df.shape)
AADT_df.columns

,FID,YEAR_,DISTRICT,COSITE,ROADWAY,DESC_FRM,DESC_TO,AADT,AADTFLG,KFLG,...,END_POST,KFCTR,K100FCTR,DFCTR,TFCTR,Shape_Leng,COUNTYDOT,COUNTY,MNG_DIST,Shape__Length
0,1,2025,7,149035,14000055,SCENIC DR,CR-1/LITTLE RD,1950,C,F,...,2.297,9.0,0,55.7,7.2,3703.3090,14,Pasco,7,3703.309013
1,2,2025,6,878337,87500500,US-1/SR-5/DIXIE HWY,END OF BRIDGE SR-821,25000,C,F,...,1.800,9.0,0,61.0,4.2,2893.9446,87,Miami-Dade,6,2893.944638
2,3,2025,7,149091,14550000,SR-54,CR577/578/SAINT JOE,19700,C,F,...,10.500,9.0,0,55.7,7.2,16859.9406,14,Pasco,7,16859.940628
3,4,2025,2,385021,38020000,W GREEN ST,NaN,11000,C,F,...,2.294,9.0,0,55.6,10.5,1427.6356,38,Taylor,2,1427.635628
4,5,2025,2,390102,39040000,SR-121,CR-199,3400,C,F,...,1.460,9.5,0,52.3,9.1,2338.5701,39,Union,2,2338.570120


(21612, 24)

Index(['FID', 'YEAR_', 'DISTRICT', 'COSITE', 'ROADWAY', 'DESC_FRM', 'DESC_TO',
       'AADT', 'AADTFLG', 'KFLG', 'K100FLG', 'DFLG', 'TFLG', 'BEGIN_POST',
       'END_POST', 'KFCTR', 'K100FCTR', 'DFCTR', 'TFCTR', 'Shape_Leng',
       'COUNTYDOT', 'COUNTY', 'MNG_DIST', 'Shape__Length'],
      dtype='object')

In [72]:
# annual average daily truck volume
TV_df = pd.read_csv('Truck_Volume.csv')
display(TV_df.head())
display(TV_df.shape)
TV_df.columns

,FID,YEAR_,DISTRICT,COSITE,ROADWAY,AADT,AADTFLG,KFLG,K100FLG,DFLG,...,K100FCTR,DFCTR,TFCTR,Shape_Leng,DESC_FRM,DESC_TO,COUNTYDOT,COUNTY,MNG_DIST,Shape__Length
0,1,2025,2,380018,38010000,11100,C,F,F,F,...,0,55.6,12.3,1328.4235,NaN,NaN,38,Taylor,2,1328.423473
1,2,2025,7,146033,14610000,51000,C,F,F,F,...,0,55.7,5.9,1612.6734,COUNTY LINE RD,SR-56,14,Pasco,7,1612.673355
2,3,2025,7,149075,14000202,850,C,F,F,F,...,0,55.7,7.2,556.1986,FIVAY RD,HUDSON AV,14,Pasco,7,556.198594
3,4,2025,7,29023,2000028,4100,C,F,F,F,...,0,52.8,9.1,1732.3796,S PLEASANT GROVE RD,S CASCADE AVE,2,Citrus,7,1732.379562
4,5,2025,2,719001,71000015,22000,F,F,F,F,...,0,52.7,1.3,4033.7424,SR-21,US-17/PARK AVE,71,Clay,2,4033.742382


(21612, 25)

Index(['FID', 'YEAR_', 'DISTRICT', 'COSITE', 'ROADWAY', 'AADT', 'AADTFLG',
       'KFLG', 'K100FLG', 'DFLG', 'TFLG', 'TruckAADT', 'BEGIN_POST',
       'END_POST', 'KFCTR', 'K100FCTR', 'DFCTR', 'TFCTR', 'Shape_Leng',
       'DESC_FRM', 'DESC_TO', 'COUNTYDOT', 'COUNTY', 'MNG_DIST',
       'Shape__Length'],
      dtype='object')

### Objective:
- Classify roadway segments into low / medium / high traffic

### Research Question:
- Which characteristics are most associated with high traffic volume in Florida

### Sources:
- https://gis-fdot.opendata.arcgis.com/datasets/fdot%3A%3Aannual-average-daily-traffic-historical-tda/about
- https://gis-fdot.opendata.arcgis.com/datasets/fdot%3A%3Atruck-volume-tda/about
- https://gis-fdot.opendata.arcgis.com/datasets/fdot%3A%3Afunctional-classification-tda/about
- https://gis-fdot.opendata.arcgis.com/datasets/fdot%3A%3Aaccess-management-tda/about
- https://gis-fdot.opendata.arcgis.com/datasets/fdot%3A%3Atraffic-signal-locations-tda/about

---

- Before joining I'd like to locate keys, overlap, mismatches, and duplicates in the above two datasets ; Annual_Average_Daily_Traffic, Functional_Classification, Truck_Volume
- Later I will join the Access_Management, Traffic_Signal_Locations, and Functional_Class

# EDA

In [128]:
# NA count for each dataframe where NA values are present
# NA count for each dataframe ; only showing columns where NA values are present
EDA_tables = [
    ('AADT_df', AADT_df),
    ('TV_df', TV_df),
]

for name, df in EDA_tables:
    na_counts = df.isna().sum()
    na_counts = na_counts[na_counts > 0]
    
    print(name, "NA values:\n")
    print(na_counts)
    print()

AADT_df NA values:

DESC_FRM    2520
DESC_TO     2635
dtype: int64

TV_df NA values:

DESC_FRM    2520
DESC_TO     2635
dtype: int64



In [130]:
# is countydot the same as county ? if so, could drop one of them based on encoding need
info_ = ['DISTRICT', 'COUNTYDOT', 'COUNTY']

print("Informational field:")
for name, EDA_df in EDA_tables:
    print(name)
    display(df[info_].head())

Informational field:
AADT_df


,DISTRICT,COUNTYDOT,COUNTY
0,2,38,Taylor
1,7,14,Pasco
2,7,14,Pasco
3,7,2,Citrus
4,2,71,Clay


TV_df


,DISTRICT,COUNTYDOT,COUNTY
0,2,38,Taylor
1,7,14,Pasco
2,7,14,Pasco
3,7,2,Citrus
4,2,71,Clay


### County & CountyDOT
- countydot is numeric match to county
- a county can have multiple districts

---

# Pre-processing decisions

### Checking years are same

In [131]:
# checking years available before filtering
print("AADT years:", sorted(AADT_df['YEAR_'].unique().tolist()))
print("TV years:", sorted(TV_df['YEAR_'].unique().tolist()))
print()

# choosing matching years ; want years that exist in both AADT and Truck Volume
aadt_years = set(AADT_df['YEAR_'].unique())
tv_years = set(TV_df['YEAR_'].unique())

matching_years = sorted(aadt_years.intersection(tv_years))
print("Matching years:", matching_years)
print()

# selecting only years that exist in both datasets ; FC has no year column
AADT_match = AADT_df.loc[AADT_df['YEAR_'].isin(matching_years)].copy()
TV_match   = TV_df.loc[TV_df['YEAR_'].isin(matching_years)].copy()

# checking table sizes after filtering
print("AADT_match shape:", AADT_match.shape)
print("TV_match shape:", TV_match.shape)

AADT years: [2025]
TV years: [2025]

Matching years: [np.int64(2025)]

AADT_match shape: (21612, 24)
TV_match shape: (21612, 25)


>currently both 2025 anyway, but do have previous years to bring in

### Locating merge key

In [132]:
# potential key ; idea is roadway + start point + end point
candidates = ['ROADWAY', 'BEGIN_POST', 'END_POST']

tables = [
    ('AADT_2025', AADT_match),
    ('TV_2025', TV_match)
]

# checking whether the same roadway ids appear across tables plus start-end match for matching trucks
# using AADT_2025 as the reference first
base_name, base_df = tables[0]
base_roadways = set(base_df['ROADWAY'].dropna().unique())

print(f"Reference table: {base_name}")
print(f"{base_name} unique roadways:", len(base_roadways))
print()

# comparing to other tables
for name, df in tables[1:]:
    roadways = set(df['ROADWAY'].dropna().unique())
    shared = base_roadways.intersection(roadways)
    only_in_base = base_roadways - roadways
    only_in_other = roadways - base_roadways
    
    print(f"{base_name} vs {name}")
    print("shared roadways:", len(shared))
    print(f"only in {base_name}:", len(only_in_base))
    print(f"only in {name}:", len(only_in_other))
    print()

Reference table: AADT_2025
AADT_2025 unique roadways: 13221

AADT_2025 vs TV_2025
shared roadways: 13221
only in AADT_2025: 0
only in TV_2025: 0



In [123]:
# roadway alone is not enough since each roadway can have multiple segments
print("AADT duplicate roadway values:", AADT_match.duplicated(subset=['ROADWAY']).sum())
print("TV duplicate roadway values:", TV_match.duplicated(subset=['ROADWAY']).sum())
print()

print("AADT duplicate segment keys:", AADT_match.duplicated(subset=candidates).sum())
print("TV duplicate segment keys:", TV_match.duplicated(subset=candidates).sum())

AADT duplicate roadway values: 8391
TV duplicate roadway values: 8391

AADT duplicate segment keys: 0
TV duplicate segment keys: 0


In [124]:
# so, checking exact segment matches between AADT and TV
aadt_tv_segment_check = AADT_match.merge(
    TV_match,
    on=candidates,
    how='outer',
    indicator=True
)

print(aadt_tv_segment_check['_merge'].value_counts())

_merge
both          21612
left_only         0
right_only        0
Name: count, dtype: int64


- Looks like viable key

---